SEFIN DEUDA PUBLICA (EXCEL) TRIMESTRAL

In [23]:
import pandas as pd
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



url="C:\\Users\\HN11133\\Desktop\\Tareas\\API\\Import_Export.xlsx"


tabla=pd.read_excel(url)
df_mensual = tabla[tabla['PERIODICIDAD'].str.lower() == 'mensual'].copy()
df_anual = tabla[tabla['PERIODICIDAD'].str.lower() == 'anual'].copy()

df_mensual["VARIACION"] = df_mensual.groupby('NOMBRE_INDICADOR')['VALOR'].pct_change().round(4)
df_anual["VARIACION"] = df_anual.groupby('NOMBRE_INDICADOR')['VALOR'].pct_change().round(4)

df_mensual.head(10)
df_largo = pd.concat([df_mensual, df_anual], ignore_index=True)
df_largo["DESCRIPCION"]="Balanza Comercial de Bienes"
df_largo["TIPO"]="Importaciones y Exportaciones"

df_largo = df_largo[["FECHA","PERIODICIDAD","DESCRIPCION","VALOR","NOMBRE_INDICADOR","TIPO","VARIACION"]]

df_largo.head(10)

,FECHA,PERIODICIDAD,DESCRIPCION,VALOR,NOMBRE_INDICADOR,TIPO,VARIACION
0,2022-01-01,Mensual,Balanza Comercial de Bienes,902.740660,Exportaciones FOB,Importaciones y Exportaciones,NaN
1,2022-02-01,Mensual,Balanza Comercial de Bienes,957.354504,Exportaciones FOB,Importaciones y Exportaciones,0.0605
2,2022-03-01,Mensual,Balanza Comercial de Bienes,1195.434665,Exportaciones FOB,Importaciones y Exportaciones,0.2487
3,2022-04-01,Mensual,Balanza Comercial de Bienes,1045.061534,Exportaciones FOB,Importaciones y Exportaciones,-0.1258
4,2022-05-01,Mensual,Balanza Comercial de Bienes,1135.814765,Exportaciones FOB,Importaciones y Exportaciones,0.0868
5,2022-06-01,Mensual,Balanza Comercial de Bienes,1189.702457,Exportaciones FOB,Importaciones y Exportaciones,0.0474
6,2022-07-01,Mensual,Balanza Comercial de Bienes,1077.045243,Exportaciones FOB,Importaciones y Exportaciones,-0.0947
7,2022-08-01,Mensual,Balanza Comercial de Bienes,1078.725700,Exportaciones FOB,Importaciones y Exportaciones,0.0016
8,2022-09-01,Mensual,Balanza Comercial de Bienes,1042.171409,Exportaciones FOB,Importaciones y Exportaciones,-0.0339
9,2022-10-01,Mensual,Balanza Comercial de Bienes,884.458942,Exportaciones FOB,Importaciones y Exportaciones,-0.1513


In [19]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'INDICADORES_MACROECONOMICOS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}

In [20]:
#-----------------------------############### INSERT ##################------------------------------------#

#df_resultado['DATE_TIME']=pd.to_datetime(df_resultado['DATE_TIME']).dt.strftime('%Y-%m-%d')
data_frame=df_largo
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')

Insertando datos: 100%|██████████| 4/4 [00:38<00:00,  9.74s/it]

Proceso de insercion completado
